# Logistic Regression and Model Evaluation – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. Logistic regression: a linear model for classification
2. Fitting by maximum likelihood (log-loss)
3. Confusion matrix and metrics
4. Classification thresholds
5. ROC curve and AUC

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Logistic regression: a linear model for classification

$$P(y = 1 \mid x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

In words: a linear **score** $\beta_0 + \beta_1 x$ is squeezed by an S-curve (the *sigmoid* $\sigma$) into a probability
between 0 and 1. With several features the score is $\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$.

- $\beta_1 > 0$: the probability rises with $x$.
- Example: $\beta_0 = -5$, $\beta_1 = 1$, $x = 6$ → score 1 → $p = 1/(1+e^{-1}) \approx 0.73$ → class 1 (if $\tau = 0.5$).

**Additional explanation – odds.** $\log\frac{p}{1-p} = \beta_0 + \beta_1 x$: the score is the log-odds. $e^{\beta_1}$ is
the factor by which the odds $p/(1-p)$ change when $x$ increases by one unit.

In [ ]:
# Function: converts a score into a probability between 0 and 1
def sigmoid(s):
    """Sigmoid function: maps a score to a probability."""
    return 1 / (1 + np.exp(-s))


# Example: beta_0 = -5, beta_1 = 1 and x = 6 (score = 1)
print('p =', round(sigmoid(-5 + 1 * 6), 3))

# Simulate 200 observations: P(y = 1) is the sigmoid of the score -5 + x
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 200)
y = (rng.uniform(size=200) < sigmoid(-5 + 1 * x)).astype(int)

# Fit the logistic regression (scikit-learn needs a table with one column)
X = x.reshape(-1, 1)
model = LogisticRegression()
model.fit(X, y)
print('Estimated beta_0:', round(model.intercept_[0], 2))
print('Estimated beta_1:', round(model.coef_[0][0], 2))

# Predicted probability of class 1 (column 1 of predict_proba)
grid = np.linspace(0, 10, 300).reshape(-1, 1)
p_grid = model.predict_proba(grid)[:, 1]

# Figure showing the data and the fitted probability curve
plt.scatter(x, y, color='black', label='data (y = 0 or 1)')
plt.plot(grid, p_grid, label='predicted probability')
plt.axhline(0.5, color='orange', linestyle='--', label='threshold 0.5')
plt.xlabel('x')
plt.ylabel('P(y = 1 | x)')
plt.legend()
plt.show()

## 2. Fitting by maximum likelihood (log-loss)

Logistic regression is fitted by **maximum likelihood**: choose $\beta$ so that the observed classes are as plausible as
possible. Maximising the likelihood is the same as minimising the average **log-loss** (cross-entropy):

$$\text{log-loss} = -\frac{1}{n}\sum_{i=1}^n \Big[y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i)\Big]$$

There is no closed-form solution – the coefficients are computed numerically (e.g. by gradient-based methods). A
confident wrong prediction ($\hat p = 0.99$ for a negative case) is penalised heavily.

In [ ]:
# Log-loss of a positive case (y = 1) for different predicted probabilities
for p_hat in [0.9, 0.6, 0.5, 0.1, 0.01]:
    print('y = 1, p_hat =', p_hat, ' loss =', round(-np.log(p_hat), 3))

## 3. Confusion matrix and metrics

| | predicted negative | predicted positive |
|---|---|---|
| **actual negative** | TN | FP |
| **actual positive** | FN | TP |

This is the orientation of scikit-learn: `confusion_matrix(y_true, y_pred)` returns `[[TN, FP], [FN, TP]]` (rows =
actual class, columns = predicted class, ordered 0, 1).

| metric | formula | question |
|---|---|---|
| accuracy | $\frac{TP + TN}{n}$ | share of all predictions that are correct |
| precision | $\frac{TP}{TP + FP}$ | of all predicted positives, how many are really positive? |
| recall (TPR, sensitivity) | $\frac{TP}{TP + FN}$ | of all real positives, how many did we find? |
| FPR | $\frac{FP}{FP + TN}$ | of all real negatives, how many were wrongly flagged? |
| F1 | $2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$ | one number balancing precision and recall |

**Caution:** with imbalanced classes, accuracy alone misleads. Example with $n = 171$ test observations:

In [ ]:
# Example confusion matrix with n = 171 test observations
TP = 50
FP = 22
FN = 24
TN = 75
n = TP + FP + FN + TN

# Metrics
accuracy = (TP + TN) / n
precision = TP / (TP + FP)
recall = TP / (TP + FN)
fpr = FP / (FP + TN)
f1 = 2 * precision * recall / (precision + recall)
print('n =', n)
print('accuracy  =', round(accuracy, 3))
print('precision =', round(precision, 3))
print('recall    =', round(recall, 3))
print('FPR       =', round(fpr, 3))
print('F1        =', round(f1, 3))

# Imbalanced classes: 1000 cases, 20 positives, always predict "negative"
print(
    'Always "negative" with 2 % positives: accuracy =',
    980 / 1000,
    ' recall = 0',
)

## 4. Classification thresholds

$$\hat y = 1 \quad \text{if} \quad \hat p(x) \ge \tau$$

- $\tau$ ↑: fewer predicted positives – recall and FPR cannot increase; precision often rises, but this is not
  guaranteed.
- $\tau$ ↓: more predicted positives – recall and FPR cannot decrease.
- The default $\tau = 0.5$ is common for probability outputs; score-based classifiers (e.g. SVM) use an analogous
  threshold.

**Rule:** pick $\tau$ from the **costs** of false positives vs. false negatives. If a false negative costs $c$ times as
much as a false positive, the cost-optimal threshold of a well-calibrated model is about $\tau^* = 1/(1+c)$.

In [ ]:
# Simulated scores of 600 negative and 400 positive cases
rng = np.random.default_rng(1)
y_true = np.concatenate([np.zeros(600), np.ones(400)])
p_hat = np.concatenate([rng.beta(3, 5, 600), rng.beta(6, 3, 400)])
neg = p_hat[y_true == 0]
pos = p_hat[y_true == 1]

# Figure showing the score distributions and the threshold 0.5
plt.hist(neg, bins=30, alpha=0.7, label='actual negatives (y = 0)')
plt.hist(pos, bins=30, alpha=0.7, label='actual positives (y = 1)')
plt.axvline(0.5, color='black', label='threshold 0.5')
plt.xlabel('predicted score')
plt.ylabel('count')
plt.title('FN: positives left of the threshold, FP: negatives right of it')
plt.legend()
plt.show()

# Recall, precision and FPR for different thresholds
taus = np.linspace(0.05, 0.95, 19)
recalls = []
precisions = []
fprs = []
for tau in taus:
    predicted_pos = p_hat >= tau
    tp = np.sum(predicted_pos & (y_true == 1))
    fp = np.sum(predicted_pos & (y_true == 0))
    recalls.append(tp / len(pos))
    precisions.append(tp / (tp + fp))
    fprs.append(fp / len(neg))

# Figure showing the metrics as a function of the threshold
plt.plot(taus, recalls, marker='o', label='recall')
plt.plot(taus, precisions, marker='o', label='precision')
plt.plot(taus, fprs, marker='o', label='FPR')
plt.xlabel('threshold')
plt.title('Metrics as a function of the threshold')
plt.legend()
plt.show()

## 5. ROC curve and AUC

**ROC curve:** for every threshold $\tau$ plot (FPR, TPR) and connect the points. Each point is one threshold; the
diagonal corresponds to a random ranking (in expectation).

**AUC** = area under the ROC curve (0.5 = random ranking, 1 = perfect). Interpretation: the **probability that a random
positive case gets a higher score than a random negative case** (ties count ½). Example: AUC = 0.95 → in 95 % of such
pairs the positive case is ranked higher.

**Additional explanation.** The AUC measures the *ranking* quality of a model and does not depend on the threshold. A
model with a high AUC can still give bad results with an unsuitable threshold – and the AUC says nothing about whether the
probabilities are well calibrated.

In [ ]:
# ROC curve and AUC of the simulated scores
fpr_curve, tpr_curve, thresholds = roc_curve(y_true, p_hat)
auc = roc_auc_score(y_true, p_hat)
print('AUC =', round(auc, 4))

# Figure showing the ROC curve and the points of three thresholds
plt.plot(fpr_curve, tpr_curve, label='model')
plt.plot([0, 1], [0, 1], 'k--', label='random ranking (AUC = 0.5)')
for tau in [0.3, 0.5, 0.7]:
    plt.scatter(
        np.mean(neg >= tau), np.mean(pos >= tau), label='threshold ' + str(tau)
    )
plt.xlabel('false positive rate (FPR)')
plt.ylabel('true positive rate (TPR)')
plt.legend()
plt.show()

# AUC = share of all (positive, negative) pairs in which the positive case
# has the higher score (ties count 1/2)
correct = 0
for score in pos:
    correct = correct + np.sum(score > neg) + 0.5 * np.sum(score == neg)
print('Share of correctly ranked pairs =', round(correct / (400 * 600), 4))

**Key messages**
- Logistic regression turns a linear score into a probability; it is fitted by maximum likelihood (log-loss).
- With imbalanced classes look at precision, recall and FPR – not only at the accuracy.
- The threshold is a business decision based on the costs of errors.
- The ROC curve shows all thresholds at once; the AUC measures how well the model ranks positives above negatives.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')